# NLP Preprocessing

From `tovima_clean.csv` builds two text columns:

- `embedding_text` — natural text, for the embedding model
- `cleaned_text` — lemmatized, stopword-filtered, for TF-IDF/NPMI

In [1]:
import pandas as pd
import numpy as np
import re
import json
import string
from collections import Counter

pd.set_option("display.max_colwidth", 80)

## 0. Load

In [2]:
df = pd.read_csv("tovima_clean.csv", encoding="utf-32", sep="\t")
df["to_lists"] = df["to_lists"].apply(json.loads)
df["to_other_recipients"] = df["to_other_recipients"].apply(json.loads)

print(df.shape)

(13692, 13)


## 1. URL stripping

In [3]:
URL_RE = re.compile(r"https?\s*:\s*/\s*/\s*\S+|www\.\S+", re.IGNORECASE)


def strip_urls(text):
    return URL_RE.sub(" ", text)

In [4]:
after = df["Message"].fillna("").apply(strip_urls)
remaining = after.str.contains("http", case=False, na=False)
print(f"{remaining.sum()} messages ({remaining.mean()*100:.1f}%) still have an http/https fragment after stripping")

122 messages (0.9%) still have an http/https fragment after stripping


## 2. Quoted reply lines

In [5]:
QUOTE_LINE_RE = re.compile(r"^\s*>+.*$", re.MULTILINE)


def strip_quoted_lines(text):
    return QUOTE_LINE_RE.sub("", text)

In [6]:
has_quote_lines = df["Message"].fillna("").str.contains(QUOTE_LINE_RE, regex=True)
print(f"{has_quote_lines.sum()} messages ({has_quote_lines.mean()*100:.1f}%) have quote lines")

939 messages (6.9%) have quote lines


## 3. Greetings and closings

In [7]:
GREETING_RE = re.compile(
    r"^\s*(αγαπητ\w*|αξιότιμ\w*|καλη(μέρα|σπέρα)|γεια\s+σας)[^\n]*\n",
    re.IGNORECASE | re.MULTILINE,
)
CLOSING_KEYWORDS_RE = re.compile(
    r"(με\s+εκτίμηση|ευχαριστ\w*|φιλικά|καλό\s+(βράδυ|απόγευμα))",
    re.IGNORECASE,
)


def strip_greeting(text, search_window=80):
    head = text[: search_window + 200]
    m = GREETING_RE.search(head)
    if m and m.start() < search_window:
        return text[m.end() :]
    return text


def strip_closing(text, tail_fraction=0.25, max_tail_chars=400):
    zone_start = max(0, int(len(text) * (1 - tail_fraction)))
    tail = text[zone_start:]
    matches = list(CLOSING_KEYWORDS_RE.finditer(tail))
    if not matches:
        return text
    last = matches[-1]
    abs_start = zone_start + last.start()
    if len(text) - abs_start > max_tail_chars:
        return text
    return text[:abs_start].rstrip()


def strip_greetings_and_closings(text):
    return strip_closing(strip_greeting(text))

## 4. Build `embedding_text`

Subject gets prepended so the embedding has the topic up front.

In [8]:
def build_embedding_text(row):
    text = row["Message"]
    if not isinstance(text, str):
        return ""
    text = strip_urls(text)
    text = strip_quoted_lines(text)
    text = strip_greetings_and_closings(text)
    return text.strip()


df["embedding_text"] = df.apply(build_embedding_text, axis=1)
df["embedding_text"] = df["Subject"].fillna("") + ". " + df["embedding_text"]

df["embedding_text"].iloc[0][:300]

'[ANNOUNCEMENTS] Ανακοίνωση για θέση PhD στην Ελβετία. Επισυνάπτεται ανακοίνωση για θέση PhD στην Ελβετία'

## 5. Lemmatization, stopwords, noise filtering

Needs `el_core_news_lg` — run locally.

In [ ]:
import spacy
from spacy.lang.el.stop_words import STOP_WORDS as GREEK_STOP_WORDS

nlp = spacy.load("el_core_news_lg")
print(f"{len(GREEK_STOP_WORDS)} stopwords loaded")

spaCy returns "σε ο" as one lemma for contractions like "στην" — split before the stopword filter.

In [10]:
def lemmatize_text(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower()
        if " " in lemma:
            tokens.extend(lemma.split())
        else:
            tokens.append(lemma)
    return tokens

In [11]:
def normalize_patterns(text):
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "ΩΡΑ", text)
    text = re.sub(r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b", "ΗΜΕΡΟΜΗΝΙΑ", text)
    text = re.sub(r"\b\d{4}[./-]\d{1,2}[./-]\d{1,2}\b", "ΗΜΕΡΟΜΗΝΙΑ", text)
    text = re.sub(r"\b(19|20)\d{2}\b", "ΗΜΕΡΟΜΗΝΙΑ", text)
    text = re.sub(r"\b(69\d{8}|2\d{9})\b", "ΤΗΛΕΦΩΝΟ", text)
    text = re.sub(r"(Αίθουσα|Αμφιθέατρο)\s*\d+", "ΑΙΘΟΥΣΑ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[A-ZΑ-Ω]{2,}\d{2,}\b", "ΜΑΘΗΜΑ", text)
    return text


BAD_SUBSTRINGS = {
    "nbsp", "mso", "font", "0pt", "pt",
    "style", "width", "padding", "border",
}
SINGLE_CHAR_RE = re.compile(r"^[\W_]*[\wά-ώα-ωΑ-Ω][\W_]*$")
NUMERIC_RE = re.compile(r"^\d+$")
PUNCT_RE = re.compile(r"^[\W_]+$")


def normalize_token(t):
    return t.strip(string.punctuation)


def is_bad_token(tok):
    t = normalize_token(tok).lower()
    if not t:
        return True
    if t in BAD_SUBSTRINGS:
        return True
    if any(b in t for b in BAD_SUBSTRINGS):
        return True
    if NUMERIC_RE.match(t):
        return True
    if PUNCT_RE.match(t):
        return True
    if SINGLE_CHAR_RE.match(t):
        return True
    if len(t) <= 2:
        return True
    return False


def clean_noise(tokens):
    return [normalize_token(t).lower() for t in tokens if not is_bad_token(t)]

## 6. English content

Keep English content words; filter English stopwords only (before lemmatization).

In [12]:
from spacy.lang.en.stop_words import STOP_WORDS as ENGLISH_STOP_WORDS

print(f"{len(ENGLISH_STOP_WORDS)} English stopwords loaded")


def strip_english_stopwords(text):
    words = text.split()
    return " ".join(w for w in words if w.lower().strip(string.punctuation) not in ENGLISH_STOP_WORDS)

326 English stopwords loaded


In [13]:
sample = df["embedding_text"].sample(n=min(1000, len(df)), random_state=1)
filtered_sample = sample.apply(strip_english_stopwords)

removed_words = []
for before, after in zip(sample, filtered_sample):
    b, a = before.split(), after.split()
    if len(b) != len(a):
        removed_words.append(len(b) - len(a))

print(f"Rows affected: {len(removed_words)} / {len(sample)}")
print(f"Total words removed: {sum(removed_words)}")

Rows affected: 384 / 1000
Total words removed: 5960


## 7. Truncation

max_tokens 1000, min_tokens 5 (after lemmatization).

In [14]:
def preprocess_for_lexical_stats(text, greek_stopwords=None, max_tokens=1000, min_tokens=5):
    text = strip_english_stopwords(text)
    text = normalize_patterns(text)
    tokens = lemmatize_text(text)

    if greek_stopwords:
        tokens = [t for t in tokens if t not in greek_stopwords]

    tokens = clean_noise(tokens)
    tokens = tokens[:max_tokens]

    if len(tokens) < min_tokens:
        return None

    return " ".join(tokens)

## 8. Run it

In [ ]:
GREEK_STOP_LEMMAS = set(GREEK_STOP_WORDS)

df["cleaned_text"] = df["embedding_text"].apply(
    lambda x: preprocess_for_lexical_stats(x, greek_stopwords=GREEK_STOP_LEMMAS)
)

n_dropped = df["cleaned_text"].isna().sum()
print(f"{n_dropped} rows dropped (< 5 tokens after lemmatization/stopwords)")

df_final = df[df["cleaned_text"].notna()].reset_index(drop=True)
print("Final shape:", df_final.shape)

## 9. Sanity check: stopword filtering

In [ ]:
test_words = ["και", "που", "του", "της", "τον", "την", "το", "οι", "τα", "των",
              "να", "θα", "από", "για", "με", "σε", "είναι", "ότι", "αυτό", "αυτή"]

stop_lemmas_check = set()
for w in test_words:
    doc = nlp(w)
    if len(doc):
        stop_lemmas_check.add(doc[0].lemma_.lower())
stop_lemmas_check |= set(test_words)

all_tokens = []
for text in df_final["cleaned_text"].dropna():
    all_tokens.extend(text.split())

freq = Counter(all_tokens)
total = sum(freq.values())

leaked = {w: freq[w] for w in stop_lemmas_check if freq.get(w, 0) > 0}
leaked_total = sum(leaked.values())
print(f"leaked function-word tokens: {leaked_total} ({leaked_total/total*100:.3f}% of corpus)")
for w, c in sorted(leaked.items(), key=lambda x: -x[1]):
    print(f"  {c:>5}  {w!r}")

In [ ]:
print("top 20 tokens in cleaned_text:")
for tok, cnt in freq.most_common(20):
    print(f"  {cnt:>6}  {tok!r}")

Top of the list: normalize_patterns() placeholders + 'tovima' etc. — exclude from TF-IDF top terms later.

## 10. Save

In [ ]:
df_out = df_final.copy()
df_out["to_lists"] = df_out["to_lists"].apply(json.dumps)
df_out["to_other_recipients"] = df_out["to_other_recipients"].apply(json.dumps)

df_out.to_csv("tovima_nlp_ready.csv", sep="\t", encoding="utf-32", index=False)
print("saved:", df_out.shape)
print(df_out.columns.tolist())